In [ ]:
%%capture
%pip install ultralytics roboflow

In [ ]:
project_path = "wcama2026-melipona-tracking"
training_name = "rt_detr"

dataset_path = "dataset"

In [ ]:
from ultralytics import RTDETR

fine_tuning = False # Takes the best result and trains it a bit more.
resume = False      # If False, it starts a new training instead of continuing from where the last one left off.

if fine_tuning:
    # Loads the weights of the best trained RT-DETR model
    model = RTDETR(f'{project_path}/{training_name}/weights/best.pt')

    results = model.train(
        data=f'{dataset_path}/data.yaml',
        cfg="configs/hyp_rtdetr.yaml",
        epochs=23,
        batch=8,
        workers=6,
        cache='disk',
        imgsz=640,
        project=project_path,
        name=training_name
    )

else:
    if resume:
        # Loads RT-DETR's "last.pt" (last saved state before interruption)
        model = RTDETR(f'{project_path}/{training_name}/weights/last.pt')
        # Resumes exactly from where it left off
        results = model.train(resume=True)

    else: # Starts a new training from scratch
        # Load a model
        model = RTDETR('rtdetr-l.pt')  # Loads the pre-trained RT-DETR Large model (there is also rtdetr-x.pt for Extra Large)

        results = model.train(
            data=f'{dataset_path}/data.yaml',
            cfg="configs/hyp_rtdetr.yaml",
            epochs=23,
            batch=12,
            workers=6,
            cache='disk',
            imgsz=640,
            project=project_path,
            name=training_name
        )


Performs the model testing with the test images.

In [ ]:
"""
Module for evaluating a trained RT-DETR model on a test dataset.
This script loads the best model weights, runs the validation pipeline 
on the 'test' split, and prints detailed performance metrics 
(mAP, Precision, Recall) both overall and per class.
"""

from ultralytics import RTDETR
import os

# --- 1. Load the trained model ---
# Note the "2" appended to training_name, matching the directory where it was reportedly saved
MODEL_PATH = f'{project_path}/{training_name}/weights/best.pt'

# Check if the model file exists before loading
if not os.path.exists(MODEL_PATH):
    print(f"Error: The model file was not found at: {MODEL_PATH}")
    print("Check if the training was completed successfully and if the path is correct.")
else:
    # Load your best model using the RTDETR class
    model = RTDETR(MODEL_PATH)
    print("Model loaded successfully!")

    # --- 2. Run validation on the TEST dataset ---
    print("Starting validation on the test dataset...")
    metrics = model.val(
        data=f'{dataset_path}/data.yaml',
        split='test',
        imgsz=640,
        plots=True,
        project=project_path,
        name=f'{training_name}2_inference'
    )
    print("Validation completed!")

    # --- 3. Access and print the metrics ---
    print("\n--- Overall Metrics ---")
    print(f"mAP50-95 (overall): {metrics.box.map:.4f}")
    print(f"mAP50 (overall):    {metrics.box.map50:.4f}")
    print(f"mAP75 (overall):    {metrics.box.map75:.4f}")

    print("\n--- Performance Metrics on the Test Dataset (per class) ---")

    names = metrics.names
    map50_95 = metrics.box.maps
    map50 = metrics.box.all_ap[:, 0]
    map75 = metrics.box.all_ap[:, 5]

    precision = metrics.box.p
    recall = metrics.box.r

    for class_id, class_name in names.items():
        idx = int(class_id)

        print(f"\nClass: {class_name} (id={idx})")
        print(f"  mAP50-95: {map50_95[idx]:.4f}")
        print(f"  mAP50:    {map50[idx]:.4f}")
        print(f"  mAP75:    {map75[idx]:.4f}")
        print(f"  Precision:{precision[idx]:.4f}")
        print(f"  Recall:   {recall[idx]:.4f}")


In [ ]:
"""
Module for running RT-DETR inference on a random sample of test images.
It loads a trained model, randomly selects a specified number of images 
from a test directory, runs predictions, and saves the visual results.
"""

from ultralytics import RTDETR
import os
import glob
import random

# --- 1. Define paths ---
MODEL_PATH = f'{project_path}/{training_name}/weights/best.pt'
TEST_IMAGES_DIR = f'{dataset_path}/test/images'
NUM_IMAGES_TO_SELECT = 100 # Set the desired number of images

# --- 2. Find all images and select a random sample ---
# Use a pattern to find common image types
image_pattern = os.path.join(TEST_IMAGES_DIR, '*.jpg')
full_image_list = glob.glob(image_pattern)

# Check if any image was found
if not full_image_list:
    print(f"No images found in the folder: {TEST_IMAGES_DIR}")
else:
    print(f"Total images found: {len(full_image_list)}")

    # Ensure we do not try to select more images than available
    if len(full_image_list) < NUM_IMAGES_TO_SELECT:
        print(f"Warning: The number of images found ({len(full_image_list)}) is less than requested ({NUM_IMAGES_TO_SELECT}).")
        print("Using all available images.")
        image_sample = full_image_list
    else:
        # Randomly select images from the complete list
        image_sample = random.sample(full_image_list, NUM_IMAGES_TO_SELECT)

    print(f"Processing a random sample of {len(image_sample)} images...")

    # --- 3. Load the model ---
    model = RTDETR(MODEL_PATH)  # <-- Instantiating the correct model

    # --- 4. Run prediction on the image sample ---
    model.predict(
        source=image_sample,
        save=True,
        imgsz=640,
        conf=0.1,  # 10% confidence threshold
        project=project_path,
        name=f'{training_name}_img_test'
    )

    print("\nSample prediction completed!")
    print(f"The {len(image_sample)} images with detections were saved in the folder: {project_path}/{training_name}_img_test")


In [ ]:
"""
Module for exporting a trained RT-DETR model to multiple formats optimized for edge devices (e.g., Raspberry Pi 5).
It handles local preparation, exports the model to ONNX, OpenVINO, and NCNN in varying precisions (FP32, FP16, INT8),
renames the outputs to avoid overwriting, and compresses everything into a ZIP file for easy extraction.
"""

from ultralytics import RTDETR
import os
import shutil

# The command with '!' works if you are running in a Jupyter Notebook / Google Colab
!rm -r /content/exported_rpi5 # Deletes the directory if it already exists.

# --- Configurations ---
original_model_path = f'{project_path}/{training_name}/weights/best.pt'

# 1. LOCAL PREPARATION
local_export_dir = '/content/exported_rpi5'
os.makedirs(local_export_dir, exist_ok=True)
local_model_path = f'{local_export_dir}/best.pt'

print("--- Moving model from Drive to local storage ---")
shutil.copy(original_model_path, local_model_path)
print("Model copied to local machine successfully!")

imgsz = [640, 640]
data_yaml_path = f'{dataset_path}/data.yaml'

print(f"\nLoading model: {local_model_path}")
model = RTDETR(local_model_path)

# Helper function to prevent files from overwriting each other
def rename_export(old_path, new_name):
    new_path = os.path.join(local_export_dir, new_name)
    # If it already exists from a previous run, delete it to avoid errors
    if os.path.exists(new_path):
        if os.path.isdir(new_path):
            shutil.rmtree(new_path)
        else:
            os.remove(new_path)
    os.rename(old_path, new_path)
    print(f"-> Saved with the correct name: {new_name}")

# ============================================
# 1. Export to ONNX (FP32)
# ============================================
print("\n--- Exporting to ONNX (FP32) ---")
# FIX HERE: Changed opset to 16
onnx_path = model.export(format='onnx', imgsz=imgsz, simplify=True, opset=16)
rename_export(onnx_path, 'onnx_model_fp32.onnx')

# ============================================
# 2. Export to ONNX (FP16)
# ============================================
print("\n--- Exporting to ONNX (FP16) ---")
# FIX HERE: Changed opset to 16
onnx_fp16_path = model.export(format='onnx', imgsz=imgsz, simplify=True, half=True, opset=16)
rename_export(onnx_fp16_path, 'onnx_model_fp16.onnx')

# ============================================
# 3. Export to OpenVINO (INT8)
# ============================================
print("\n--- Exporting to OpenVINO (INT8) ---")
openvino_int8_path = model.export(format='openvino', imgsz=imgsz, int8=True, data=data_yaml_path)
rename_export(openvino_int8_path, 'int8_openvino_model')

# ============================================
# 4. Export to OpenVINO (FP16)
# ============================================
print("\n--- Exporting to OpenVINO (FP16) ---")
openvino_fp16_path = model.export(format='openvino', imgsz=imgsz, half=True)
rename_export(openvino_fp16_path, 'fp16_openvino_model')

# ============================================
# 5. Export to OpenVINO (FP32)
# ============================================
print("\n--- Exporting to OpenVINO (FP32) ---")
openvino_fp32_path = model.export(format='openvino', imgsz=imgsz)
rename_export(openvino_fp32_path, 'fp32_openvino_model')

# ============================================
# 6. Export to NCNN (FP32)
# ============================================
print("\n--- Exporting to NCNN (FP32) ---")
ncnn_fp32_path = model.export(format='ncnn', imgsz=imgsz)
rename_export(ncnn_fp32_path, 'ncnn_fp32')

# ============================================
# 7. Export to NCNN (FP16)
# ============================================
print("\n--- Exporting to NCNN (FP16) ---")
ncnn_fp16_path = model.export(format='ncnn', imgsz=imgsz, half=True)
rename_export(ncnn_fp16_path, 'ncnn_fp16')

# ============================================
# 8. Compress and Move to Drive
# ============================================
print("\n--- Compressing exported models ---")
zip_base_name = '/content/exported_models_rpi5'

# Compress everything inside 'local_export_dir'
shutil.make_archive(zip_base_name, 'zip', local_export_dir)
print(f"ZIP file created locally at: {zip_base_name}.zip")

print("\n--- Moving ZIP file to Google Drive ---")
drive_dest_dir = f'{project_path}/{training_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

drive_zip_path = f'{drive_dest_dir}/exported_rpi5.zip'
shutil.copy(f'{zip_base_name}.zip', drive_zip_path)

print(f"\n✅ SUCCESS! Models generated with unique names and saved in Drive at: {drive_zip_path}")


In [ ]:
"""
Module for testing and comparing different exported formats of an RT-DETR model 
(PyTorch, ONNX, OpenVINO). It runs inference on both GPU and CPU to extract 
speed metrics (ms/image) and accuracy metrics (mAP, Precision, Recall), 
compiling the results into a final CSV file.
"""

import os
import shutil
import pandas as pd
from ultralytics import RTDETR

dataset_path = "/content/Abelhas-V3-1"
local_export_dir = '/content/exportados_rpi5'

project_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_rt_dert"
training_name = "rt_dert_28_02_20262"

# ==========================================
# 1. Define the models with the correct names
# ==========================================
models_to_test = {
    "PyTorch_Original": f"{local_export_dir}/best.pt",
    "ONNX_FP32": f"{local_export_dir}/onnx_model_fp32.onnx",
    "ONNX_FP16": f"{local_export_dir}/onnx_model_fp16.onnx",
    "OpenVINO_FP32": f"{local_export_dir}/fp32_openvino_model",
    "OpenVINO_FP16": f"{local_export_dir}/fp16_openvino_model",
    "OpenVINO_INT8": f"{local_export_dir}/int8_openvino_model"
}

# ==========================================
# 2. Run the tests and extract metrics
# ==========================================
results = []

for model_name, model_path in models_to_test.items():
    if not os.path.exists(model_path):
        print(f"⚠️ Skipping {model_name}: File/Folder not found at {model_path}")
        continue

    print(f"\n========================================")
    print(f"🚀 Starting test: {model_name}")
    print(f"========================================")

    try:
        # Instantiate the model using the RTDETR class
        model = RTDETR(model_path)

        gpu_time = None
        cpu_time = None
        base_metrics = None

        # --- GPU Test ---
        print("   -> Testing GPU inference...")
        try:
            metrics_gpu = model.val(
                data=f'{dataset_path}/data.yaml',
                split='test',
                imgsz=640,
                device=0,  # Forces the use of GPU
                plots=False,
                verbose=False
            )
            gpu_time = round(metrics_gpu.speed['inference'], 2)
            base_metrics = metrics_gpu # Stores the main metrics
            print(f"      GPU Time: {gpu_time} ms/image")
        except Exception as e:
            print(f"      ⚠️ GPU not supported or failed for format {model_name}. Error: {e}")

        # --- CPU Test ---
        print("   -> Testing CPU inference...")
        try:
            metrics_cpu = model.val(
                data=f'{dataset_path}/data.yaml',
                split='test',
                imgsz=640,
                device='cpu',  # Forces the use of CPU
                plots=False,
                verbose=False
            )
            cpu_time = round(metrics_cpu.speed['inference'], 2)
            if base_metrics is None:
                base_metrics = metrics_cpu # If GPU failed, uses CPU accuracy
            print(f"      CPU Time: {cpu_time} ms/image")
        except Exception as e:
            print(f"      ⚠️ CPU failed for format {model_name}. Error: {e}")


        # --- Extract Accuracy Metrics ---
        # Only proceeds if it successfully ran on CPU or GPU
        if base_metrics is not None:
            names = base_metrics.names
            map50_95 = base_metrics.box.maps
            precision = base_metrics.box.p
            recall = base_metrics.box.r
            map50_classes = base_metrics.box.all_ap[:, 0]
            map75_classes = base_metrics.box.all_ap[:, 5]

            for class_id, class_name in names.items():
                idx = int(class_id)
                results.append({
                    "Model": model_name,
                    "Class": class_name,
                    "Precision": round(precision[idx], 4),
                    "Recall": round(recall[idx], 4),
                    "mAP50": round(map50_classes[idx], 4),
                    "mAP75": round(map75_classes[idx], 4),
                    "mAP50-95": round(map50_95[idx], 4),
                    "Inference_GPU (ms)": gpu_time if gpu_time else "N/A",
                    "Inference_CPU (ms)": cpu_time if cpu_time else "N/A"
                })

            print(f"✅ {model_name} tested successfully!")
        else:
            print(f"❌ Total failure when testing {model_name}. No metrics extracted.")

    except Exception as e:
        print(f"❌ Critical error when testing {model_name}: {e}")

# ==========================================
# 3. Generate and Save the CSV
# ==========================================
print("\n--- Compiling results into CSV ---")
df_results = pd.DataFrame(results)
print(df_results.head(12))

csv_path = '/content/models_comparison.csv'
df_results.to_csv(csv_path, index=False)

# Ensure the destination folder exists on Drive before copying
drive_dest_dir = f'{project_path}/{training_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

# Copies the finalized CSV file to Google Drive
drive_csv_path = f'{drive_dest_dir}/models_comparison.csv'
shutil.copy(csv_path, drive_csv_path)

print(f"\n✅ All done! CSV saved in your Drive at: {drive_csv_path}")